<a href="https://colab.research.google.com/github/nmanthas/AgenticAI/blob/autonomous-IT-support-agent/SG_Reflex_Assignment_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment: The Autonomous IT Support Agent**

This is a coding assignment for the **Level 1 IT Incident Responder**.

This assignment assumes the learners have the `openai` library installed and an API key ready and stored in the variable name `OPENAI_APIKEY` as a notebook secret variable and enabled

---

**Objective:** You are building an AI agent that acts as the "first responder" for server incidents. It must:

1. **Investigate:** Check server health and logs when a user reports an issue.
2. **Act:** If CPU is critical (>90%), it should **Restart** the service.
3. **Escalate:** If the issue is complex or logs show "Payment Gateway Error", it should **Escalate** to a human.

In [1]:
import os
import json
from openai import OpenAI
from google.colab import userdata

In [3]:
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# Initialize Client

==========================================
## PART 1: DEFINE THE TOOLS (BUSINESS LOGIC)
==========================================

In [4]:
# --- Already implement tool 1: Check Health ---
def get_server_health(server_id: str) -> str:
    """Returns CPU and Memory usage for a given server."""
    print(f"-> TOOL: Checking health for {server_id}...")

    metrics = {
        # Scenario 1: High CPU (Needs Restart)
        "payment-server-01": {"cpu": "98%", "memory": "40%", "status": "Warning"},

        # Scenario 2: Healthy (No Action Needed)
        "db-node-02": {"cpu": "12%", "memory": "60%", "status": "Healthy"},

        # Scenario 3: High Memory Leak (Needs Restart or Escalation)
        "auth-service-03": {"cpu": "45%", "memory": "95%", "status": "Critical"},

        # Scenario 4: Network/Dependency Failure (Needs Escalation)
        "search-index-09": {"cpu": "10%", "memory": "15%", "status": "Error"},

        # Scenario 5: Completely Normal
        "frontend-node-04": {"cpu": "25%", "memory": "30%", "status": "Healthy"},
    }

    result = metrics.get(server_id, {"error": "Server not found. Check the ID."})
    return json.dumps(result)


In [5]:
def fetch_recent_logs(server_id: str, lines: int = 5) -> str:
    """Returns the last N lines of logs."""
    print(f"-> TOOL: Fetching last {lines} log lines for {server_id}...")

    # Different logs for different servers to trigger different agent behaviors
    log_database = {
        "payment-server-01": [
            "[INFO] Request received /pay/v1",
            "[WARN] CPU threshold exceeded 90%",
            "[WARN] Thread pool exhaustion",
            "[CRITICAL] Process hung, not accepting new connections",
            "[ERROR] Timeout waiting for thread"
        ],
        "db-node-02": [
            "[INFO] Backup started",
            "[INFO] Backup completed successfully",
            "[INFO] User query executed in 12ms",
            "[INFO] Health check: OK",
            "[INFO] Replication sync active"
        ],
        "auth-service-03": [
            "[INFO] Token validated user_882",
            "[WARN] Garbage collection taking too long (>5s)",
            "[ERROR] java.lang.OutOfMemoryError: Java heap space",
            "[CRITICAL] Application crashing due to memory leak",
            "[INFO] Restarting context..."
        ],
        "search-index-09": [
            "[INFO] Indexing started",
            "[ERROR] Connection refused: elastic-cluster-main:9200",
            "[ERROR] Failed to write document ID 4432",
            "[CRITICAL] Dependency Unreachable: Search Engine is down",
            "[ERROR] Retrying in 30s..."
        ],
        "frontend-node-04": [
            "[INFO] GET /home 200 OK",
            "[INFO] GET /assets/logo.png 200 OK",
            "[INFO] GET /login 200 OK",
            "[INFO] GET /api/v1/status 200 OK",
            "[INFO] Health check passed"
        ]
    }

    # Default logs if server not found in specific list
    default_logs = ["[INFO] System stable", "[INFO] Heartbeat signal received"]

    logs = log_database.get(server_id, default_logs)
    return json.dumps({"logs": logs[:lines]})

In [6]:
# --- SOLUTION TASK 1: Implement the Restart Tool ---
def restart_service(server_id: str) -> str:
    """Simulates a service restart."""
    print(f"-> TOOL: Restarting service {server_id}...")

    # In a real scenario, this would run a subprocess command or API call
    result = {
        "server_id": server_id,
        "status": "success",
        "message": "Service restart command issued successfully."
    }
    return json.dumps(result)

In [7]:
# --- SOLUTION TASK 2: Implement the Escalation Tool ---
def escalate_to_engineer(summary: str) -> str:
    """Simulates sending an alert to a human."""
    print(f"-> TOOL: Escalating to human engineer. Reason: {summary}")

    # In a real scenario, this would send a Slack message or PagerDuty alert
    result = {
        "status": "escalated",
        "ticket_id": "INC-999",
        "assigned_to": "On-Call Engineer"
    }
    return json.dumps(result)

In [8]:
# Map functions for the agent execution loop
AVAILABLE_FUNCTIONS = {
    "get_server_health": get_server_health,
    "fetch_recent_logs": fetch_recent_logs,
    "restart_service": restart_service,
    "escalate_to_engineer": escalate_to_engineer,
}

==========================================
## PART 2: DEFINE THE AGENT SCHEMA
==========================================


In [9]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_server_health",
            "description": "Checks the current CPU and memory usage of a specific server.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server, e.g., 'payment-server-01'"}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_recent_logs",
            "description": "Retrieves the most recent log entries from a server to diagnose errors.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server."},
                    "lines": {"type": "integer", "description": "Number of log lines to fetch."}
                },
                "required": ["server_id"]
            }
        }
    },
    # --- SOLUTION TASK 3: Define Schema for restart_service ---
    {
        "type": "function",
        "function": {
            "name": "restart_service",
            "description": "Restarts a specific server service. Use this when CPU usage is critically high or the process is unresponsive.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server to restart."}
                },
                "required": ["server_id"]
            }
        }
    },
    # --- SOLUTION TASK 4: Define Schema for escalate_to_engineer ---
    {
        "type": "function",
        "function": {
            "name": "escalate_to_engineer",
            "description": "Escalates the issue to a human engineer.Use this when automated fixes fail, or when the error logs indicate a complex issue like a payment gateway failure.",
            "parameters": {
                "type": "object",
                "properties": {
                     "summary": {"type": "string", "description": "A brief summary of the findings (health status, log errors) and why you are escalating."}
                },
                "required": ["summary"]
            }
        }
    }
]


 ==========================================
## PART 3: THE AGENT EXECUTION LOOP
 ==========================================


In [12]:
def run_it_agent(user_issue: str):
    print(f"\n--- New Incident: {user_issue} ---")

    messages = [
        {"role": "system", "content": "You are a Level 1 IT Responder. Investigate server issues. "
                                      "If CPU or Memory is > 90%, restart the service. If logs show critical dependency errors (like connection refused) that a restart won't fix, escalate to an engineer."},
        {"role": "user", "content": user_issue}
    ]

    while True:
        print("\n[AI Thinking...]")
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        response_msg = response.choices[0].message
        messages.append(response_msg)

        if response_msg.tool_calls:
            for tool_call in response_msg.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # Retrieve the actual python function based on name
                function_to_call = AVAILABLE_FUNCTIONS.get(func_name)

                if function_to_call:
                    # Execute the function
                    tool_output = function_to_call(**func_args)

                    # --- SOLUTION TASK 5: Append the result to messages ---
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,  # CRITICAL: Links output to the request
                        "name": func_name,
                        "content": tool_output
                    })

        else:
            print(f"\n[FINAL RESPONSE]: {response_msg.content}")
            break

 ==========================================
## PART 4: TEST SCENARIOS
 ==========================================


In [13]:
# Scenario A: Should trigger a restart (CPU is 98%)
run_it_agent("The payment-server-01 is extremely slow and timing out.")


--- New Incident: The payment-server-01 is extremely slow and timing out. ---

[AI Thinking...]
-> TOOL: Checking health for payment-server-01...
-> TOOL: Fetching last 10 log lines for payment-server-01...

[AI Thinking...]
-> TOOL: Restarting service payment-server-01...

[AI Thinking...]

[FINAL RESPONSE]: The payment-server-01 was experiencing high CPU usage at 98% and issues such as process hangs and thread pool exhaustion, leading to timeouts. I have successfully restarted the service to address these problems. Please check the server's performance now. If further issues persist, let me know!


In [14]:
# Scenario B: Should trigger an escalation (DB is healthy but logs might be weird)
run_it_agent("Something is wrong with db-node-02")


--- New Incident: Something is wrong with db-node-02 ---

[AI Thinking...]
-> TOOL: Checking health for db-node-02...
-> TOOL: Fetching last 10 log lines for db-node-02...

[AI Thinking...]

[FINAL RESPONSE]: The server `db-node-02` is reporting as healthy:

- **CPU Usage:** 12%
- **Memory Usage:** 60%
- **Status:** Healthy

The recent logs show normal operations:
- Backup process started and completed successfully.
- User queries are executing with normal response times.
- Health checks are passing, and replication sync is active.

Currently, there are no indicators of a problem with `db-node-02`. If there are specific issues still being experienced, they may not be reflected in the current health metrics or logs. Please provide more details or re-evaluate the situation if necessary.


In [15]:
# Scenario C: The High Memory Case (auth-service-03)
# Agent should see Memory 95% + OutOfMemoryError logs -> Restart
run_it_agent("Users are reporting login failures on auth-service-03.")

print("\n" + "="*50 + "\n")


--- New Incident: Users are reporting login failures on auth-service-03. ---

[AI Thinking...]
-> TOOL: Checking health for auth-service-03...
-> TOOL: Fetching last 10 log lines for auth-service-03...

[AI Thinking...]
-> TOOL: Restarting service auth-service-03...

[AI Thinking...]

[FINAL RESPONSE]: The auth-service-03 had a critical memory issue with 95% memory usage and logs indicating an "OutOfMemoryError". The service has been successfully restarted to resolve this problem. Please monitor the service to ensure it stabilizes, and report back if there are continued issues.




In [16]:
# Scenario D: The Dependency Failure (search-index-09)
# Agent should see healthy CPU but "Connection Refused" logs -> Escalate
run_it_agent("Search isn't working. Can you check search-index-09?")

print("\n" + "="*50 + "\n")


--- New Incident: Search isn't working. Can you check search-index-09? ---

[AI Thinking...]
-> TOOL: Checking health for search-index-09...
-> TOOL: Fetching last 20 log lines for search-index-09...

[AI Thinking...]
-> TOOL: Escalating to human engineer. Reason: Server search-index-09 is experiencing a 'Connection refused' error to the elastic-cluster-main at port 9200, as indicated by recent logs. This is causing search indexing to fail. The CPU and memory usage are well within normal ranges, but the critical dependency on the search engine needs to be addressed by an engineer.

[AI Thinking...]

[FINAL RESPONSE]: The issue with server **search-index-09** has been escalated to an engineer due to a critical dependency error: "Connection refused" to the elastic-cluster-main. A ticket (ID: **INC-999**) has been created and assigned to the on-call engineer. They will investigate further to resolve this issue.




In [17]:
# Scenario E: The Healthy Server (frontend-node-04)
# Agent should see normal stats and 200 OK logs -> Do nothing / Report healthy
run_it_agent("Check frontend-node-04 just to be safe.")


--- New Incident: Check frontend-node-04 just to be safe. ---

[AI Thinking...]
-> TOOL: Checking health for frontend-node-04...
-> TOOL: Fetching last 10 log lines for frontend-node-04...

[AI Thinking...]

[FINAL RESPONSE]: The server "frontend-node-04" is functioning normally with a CPU usage of 25% and memory usage of 30%, indicating it is in a healthy state. Recent logs show normal activity with all operations returning a status of 200 OK. No further actions are needed.
